In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from tqdm import tqdm

In [9]:
df = pd.read_csv(
    "dataset/champs_elysees.csv", sep=";", parse_dates=["Date et heure de comptage"]
)
df.head()

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,4264,AV_Champs_Elysees,2024-12-09 05:00:00+01:00,199.0,2.20945,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
1,4264,AV_Champs_Elysees,2024-12-09 06:00:00+01:00,235.0,2.28778,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
2,4264,AV_Champs_Elysees,2024-12-09 09:00:00+01:00,1041.0,11.63222,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
3,4264,AV_Champs_Elysees,2025-09-02 09:00:00+02:00,1139.0,28.39222,Pré-saturé,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Ouvert,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."
4,4264,AV_Champs_Elysees,2024-09-04 20:00:00+02:00,686.0,13.21611,Fluide,2294,Av_Champs_Elysees-Washington,2293,Av_Champs_Elysees-Berri,Invalide,1996-10-10,2023-01-01,"48.87153587897718, 2.3017227924560624","{""coordinates"": [[2.3009951475338775, 48.87177..."


In [10]:
df.isna().sum()

Identifiant arc                0
Libelle                        0
Date et heure de comptage      0
Débit horaire                563
Taux d'occupation            622
Etat trafic                    0
Identifiant noeud amont        0
Libelle noeud amont            0
Identifiant noeud aval         0
Libelle noeud aval             0
Etat arc                       0
Date debut dispo data          0
Date fin dispo data            0
geo_point_2d                   0
geo_shape                      0
dtype: int64

In [11]:
fig = go.Figure()
files = [
    "dataset/champs_elysees.csv",
    "dataset/sts_peres.csv",
    "dataset/convention.csv",
]
for file in files:
    tmp = pd.read_csv(file, parse_dates=["Date et heure de comptage"], sep=";")
    tmp = tmp.sort_values(by="Date et heure de comptage")
    fig.add_trace(
        go.Scatter(
            x=tmp["Date et heure de comptage"],
            y=tmp["Débit horaire"],
            mode="lines",
            name=file,
            hovertemplate="%{x}<br>Débit horaire: %{y}<extra></extra>",
        )
    )

fig.update_layout(
    title="Débit horaire vs Date",
    xaxis_title="Date et heure de comptage",
    yaxis_title="Débit horaire",
    legend_title="Fichier",
)
fig.show()

## Vacances, jours fériés et Paris Respire

In [12]:
from utils.off_days import add_categorical_day_columns

df = add_categorical_day_columns(df)

## XGBoost Model

In [14]:
fig = go.Figure()
file = "dataset/champs_elysees.csv"
df = pd.read_csv(file, parse_dates=["Date et heure de comptage"], sep=";")
df = add_categorical_day_columns(df)
df = df.sort_values(by="Date et heure de comptage")

# plot full series in blue
fig.add_trace(
    go.Scatter(
        x=df["Date et heure de comptage"],
        y=df["Débit horaire"],
        mode="lines",
        name=file.split("/")[-1].replace(".csv", ""),
        line=dict(color="blue"),
        hovertemplate="%{x}<br>Débit horaire: %{y}<extra></extra>",
    )
)

# overlay holiday points in red
holidays = df[df["is_holiday"] == 1]
fig.add_trace(
    go.Scatter(
        x=holidays["Date et heure de comptage"],
        y=holidays["Débit horaire"],
        mode="markers",
        name=f"{file.split("/")[-1].replace(".csv", "")} - holiday",
        marker=dict(color="red", size=6),
        hovertemplate="%{x}<br>Débit horaire: %{y}<extra></extra>",
    )
)
off_days = df[df["is_off_day"] == 1]
fig.add_trace(
    go.Scatter(
        x=off_days["Date et heure de comptage"],
        y=off_days["Débit horaire"],
        mode="markers",
        name=f"{file.split("/")[-1].replace(".csv", "")} - off day",
        marker=dict(color="purple", size=6),
        hovertemplate="%{x}<br>Débit horaire: %{y}<extra></extra>",
    )
)
paris_respire_days = df[df["is_paris_respire"] == 1]
fig.add_trace(
    go.Scatter(
        x=paris_respire_days["Date et heure de comptage"],
        y=paris_respire_days["Débit horaire"],
        mode="markers",
        name=f"{file.split("/")[-1].replace(".csv", "")} - Paris Respire",
        marker=dict(color="green", size=6),
        hovertemplate="%{x}<br>Débit horaire: %{y}<extra></extra>",
    )
)

In [16]:
from utils.process_lags import process_lags

N_day_lags = 10

df, Lags = process_lags(df, N_day_lags)
df[Lags].tail()

/Users/thibault/Documents/GitHub/trafic_prediction/utils/process_lags.py:13: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

/Users/thibault/Documents/GitHub/trafic_prediction/utils/process_lags.py:21: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/Users/thibault/Documents/GitHub/trafic_prediction/utils/process_lags.py:21: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/Users/thibault/Documents/GitHub/trafic_prediction/utils/process_lags.py:21: PerformanceWarning:

DataFrame is 

,lag_1_0,lag_1_1,lag_1_2,lag_1_3,lag_1_4,lag_1_5,lag_1_6,lag_1_7,lag_1_8,lag_1_9,...,lag_10_14,lag_10_15,lag_10_16,lag_10_17,lag_10_18,lag_10_19,lag_10_20,lag_10_21,lag_10_22,lag_10_23
10168,1010.0,905.0,908.0,943.0,903.0,1057.0,1068.0,952.0,996.0,887.0,...,316.0,526.0,642.0,564.0,639.0,525.0,667.0,904.0,941.0,960.0
10169,874.0,1010.0,905.0,908.0,943.0,903.0,1057.0,1068.0,952.0,996.0,...,280.0,316.0,526.0,642.0,564.0,639.0,525.0,667.0,904.0,941.0
10170,853.0,874.0,1010.0,905.0,908.0,943.0,903.0,1057.0,1068.0,952.0,...,317.0,280.0,316.0,526.0,642.0,564.0,639.0,525.0,667.0,904.0
10171,689.0,853.0,874.0,1010.0,905.0,908.0,943.0,903.0,1057.0,1068.0,...,454.0,317.0,280.0,316.0,526.0,642.0,564.0,639.0,525.0,667.0
10172,622.0,689.0,853.0,874.0,1010.0,905.0,908.0,943.0,903.0,1057.0,...,605.0,454.0,317.0,280.0,316.0,526.0,642.0,564.0,639.0,525.0


In [17]:
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import joblib


# prepare data
df_ml = df.copy()
df_ml["Date et heure de comptage"] = pd.to_datetime(
    df_ml["Date et heure de comptage"], utc=True
)
print(df_ml["Date et heure de comptage"].isna().sum())
df_ml = df_ml.reset_index(drop=True).sort_values(by="Date et heure de comptage")

# handle missing target
df_ml = df_ml[~df_ml["Débit horaire"].isna()]

# feature engineering: temporal features + existing numeric/categorical
df_ml["cos_hour"] = np.cos(df_ml["Date et heure de comptage"].dt.hour / 24 * 2 * np.pi)
df_ml["cos_dayofweek"] = np.cos(
    df_ml["Date et heure de comptage"].dt.dayofweek / 7 * 2 * np.pi
)
df_ml["cos_month"] = np.cos(
    df_ml["Date et heure de comptage"].dt.month / 12 * 2 * np.pi
)
df_ml["sin_hour"] = np.sin(df_ml["Date et heure de comptage"].dt.hour / 24 * 2 * np.pi)
df_ml["sin_dayofweek"] = np.sin(
    df_ml["Date et heure de comptage"].dt.dayofweek / 7 * 2 * np.pi
)
df_ml["sin_month"] = np.sin(
    df_ml["Date et heure de comptage"].dt.month / 12 * 2 * np.pi
)

# select features
feat_num = [
    "cos_hour",
    "cos_dayofweek",
    "cos_month",
    "sin_hour",
    "sin_dayofweek",
    "sin_month",
    "is_holiday",
    "is_off_day",
]
feat_cat = []  # label-encode these categorical cols
features = feat_num + feat_cat + Lags

# fill/encode
# df_ml["Taux d'occupation"] = df_ml["Taux d'occupation"].fillna(
#     df_ml["Taux d'occupation"].median()
# )
for c in feat_cat:
    df_ml[c] = df_ml[c].fillna("NA")
    le = LabelEncoder()
    df_ml[c] = le.fit_transform(df_ml[c].astype(str))

X = df_ml[features]
y = df_ml["Débit horaire"]

# train/valid split (80/20)
N_train = int(0.8 * len(df_ml))
X_train = X[:N_train]
y_train = y[:N_train]
X_valid = X[N_train:]
y_valid = y[N_train:]

print(f"Training samples: {len(X_train)}, features: {X_train.shape[1]}")
# XGBoost regressor (sklearn API)
model = XGBRegressor(
    n_estimators=500,
    max_depth=20,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    colsample_bytree=0.1,
    rowsample=0.5,
)

# set eval metric on the model (don't pass eval_metric to fit)
model.set_params(eval_metric="rmse")
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=True,
)

# evaluation
preds = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, preds))
print(f"Validation RMSE: {rmse:.2f}")

# save model
joblib.dump(model, "xgb_debit_horaire.joblib")

0
Training samples: 6962, features: 248


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning:

[11:20:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "rowsample" } are not used.




[0]	validation_0-rmse:291.81352	validation_1-rmse:244.64837
[1]	validation_0-rmse:279.12937	validation_1-rmse:236.46452
[2]	validation_0-rmse:267.12964	validation_1-rmse:228.99504
[3]	validation_0-rmse:255.74742	validation_1-rmse:221.97569
[4]	validation_0-rmse:244.63923	validation_1-rmse:215.14649
[5]	validation_0-rmse:234.14177	validation_1-rmse:208.33836
[6]	validation_0-rmse:224.10234	validation_1-rmse:202.51242
[7]	validation_0-rmse:214.74167	validation_1-rmse:196.84113
[8]	validation_0-rmse:205.56107	validation_1-rmse:191.31559
[9]	validation_0-rmse:196.85685	validation_1-rmse:186.90450
[10]	validation_0-rmse:188.43870	validation_1-rmse:182.53421
[11]	validation_0-rmse:180.71292	validation_1-rmse:178.28118
[12]	validation_0-rmse:173.11823	validation_1-rmse:174.29171
[13]	validation_0-rmse:165.84347	validation_1-rmse:170.59926
[14]	validation_0-rmse:159.00089	validation_1-rmse:167.44300
[15]	validation_0-rmse:152.32222	validation_1-rmse:164.11182
[16]	validation_0-rmse:146.02148	v

['xgb_debit_horaire.joblib']

In [26]:
print(X_train.shape, y_train.shape, X_valid.shape, y_valid.shape, X.shape, y.shape)

(6962, 248) (6962,) (1741, 248) (1741,) (8703, 248) (8703,)


In [27]:
X_train

,cos_hour,cos_dayofweek,cos_month,sin_hour,sin_dayofweek,sin_month,is_holiday,is_off_day,lag_1_0,lag_1_1,...,lag_10_14,lag_10_15,lag_10_16,lag_10_17,lag_10_18,lag_10_19,lag_10_20,lag_10_21,lag_10_22,lag_10_23
0,7.071068e-01,0.623490,-1.836970e-16,0.707107,-0.781831,-1.000000,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5.000000e-01,0.623490,-1.836970e-16,0.866025,-0.781831,-1.000000,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.588190e-01,0.623490,-1.836970e-16,0.965926,-0.781831,-1.000000,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6.123234e-17,0.623490,-1.836970e-16,1.000000,-0.781831,-1.000000,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-2.588190e-01,0.623490,-1.836970e-16,0.965926,-0.781831,-1.000000,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8346,7.071068e-01,-0.900969,-5.000000e-01,-0.707107,0.433884,-0.866025,1.0,0.0,660.0,649.0,...,868.0,602.0,361.0,225.0,150.0,207.0,390.0,498.0,591.0,626.0
8347,8.660254e-01,-0.900969,-5.000000e-01,-0.500000,0.433884,-0.866025,1.0,1.0,608.0,660.0,...,751.0,868.0,602.0,361.0,225.0,150.0,207.0,390.0,498.0,591.0
8348,9.659258e-01,-0.900969,-5.000000e-01,-0.258819,0.433884,-0.866025,1.0,1.0,560.0,608.0,...,844.0,751.0,868.0,602.0,361.0,225.0,150.0,207.0,390.0,498.0
8349,1.000000e+00,-0.900969,-5.000000e-01,0.000000,-0.433884,-0.866025,1.0,1.0,434.0,560.0,...,882.0,844.0,751.0,868.0,602.0,361.0,225.0,150.0,207.0,390.0


In [28]:
# combine predictions with validation ground truth and timestamps, then plot
pred_series_val = pd.Series(preds, index=X_valid.index, name="pred")
pred_series_train = pd.Series(model.predict(X_train), index=X_train.index, name="pred")

plot_df_val = (
    pd.DataFrame(
        {
            "date": df_ml.loc[X_valid.index, "Date et heure de comptage"],
            "actual": y_valid,
            "pred": pred_series_val,
        }
    )
    .sort_values("date")
    .reset_index(drop=True)
)
plot_df_train = (
    pd.DataFrame(
        {
            "date": df_ml.loc[X_train.index, "Date et heure de comptage"],
            "actual": y_train,
            "pred": pred_series_train,
        }
    )
    .sort_values("date")
    .reset_index(drop=True)
)

fig_pred = go.Figure()
fig_pred.add_trace(
    go.Scatter(
        x=plot_df_val["date"],
        y=plot_df_val["actual"],
        mode="lines+markers",
        name="VAL - Actual Débit horaire",
        line=dict(color="blue"),
        hovertemplate="%{x}<br>Actual: %{y}<extra></extra>",
    )
)
fig_pred.add_trace(
    go.Scatter(
        x=plot_df_val["date"],
        y=plot_df_val["pred"],
        mode="lines+markers",
        name="VAL - Predicted Débit horaire",
        line=dict(color="red"),
        hovertemplate="%{x}<br>Predicted: %{y:.1f}<extra></extra>",
    )
)
fig_pred.add_trace(
    go.Scatter(
        x=plot_df_train["date"],
        y=plot_df_train["actual"],
        mode="lines+markers",
        name="TRAIN - Actual Débit horaire",
        line=dict(color="lightblue"),
        hovertemplate="%{x}<br>Actual: %{y}<extra></extra>",
    )
)
fig_pred.add_trace(
    go.Scatter(
        x=plot_df_train["date"],
        y=plot_df_train["pred"],
        mode="lines+markers",
        name="TRAIN - Predicted Débit horaire",
        line=dict(color="lightcoral"),
        hovertemplate="%{x}<br>Predicted: %{y:.1f}<extra></extra>",
    )
)

fig_pred.update_layout(
    title="Actual vs Predicted Débit horaire (validation set) over time",
    xaxis_title="Date et heure de comptage",
    yaxis_title="Débit horaire",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_pred.show()

In [29]:
df_ml.head()

,Date et heure de comptage,Identifiant arc,Libelle,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,cos_month,sin_hour,sin_dayofweek,sin_month,date,is_sunday,month,year,error_abs,error
0,2024-09-01 03:00:00+00:00,4264.0,AV_Champs_Elysees,532.0,8.08889,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,-1.836970e-16,0.707107,-0.781831,-1.0,2024-09-01,1,9,2024,146.165619,-146.165619
1,2024-09-01 04:00:00+00:00,4264.0,AV_Champs_Elysees,331.0,3.91500,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,-1.836970e-16,0.866025,-0.781831,-1.0,2024-09-01,1,9,2024,23.439850,-23.439850
2,2024-09-01 05:00:00+00:00,4264.0,AV_Champs_Elysees,272.0,2.69167,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,-1.836970e-16,0.965926,-0.781831,-1.0,2024-09-01,1,9,2024,18.904236,18.904236
3,2024-09-01 06:00:00+00:00,4264.0,AV_Champs_Elysees,191.0,2.23612,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,-1.836970e-16,1.000000,-0.781831,-1.0,2024-09-01,1,9,2024,35.761581,35.761581
4,2024-09-01 07:00:00+00:00,4264.0,AV_Champs_Elysees,201.0,2.63333,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,-1.836970e-16,0.965926,-0.781831,-1.0,2024-09-01,1,9,2024,86.811035,86.811035


In [30]:
df_ml.tail()

,Date et heure de comptage,Identifiant arc,Libelle,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,cos_month,sin_hour,sin_dayofweek,sin_month,date,is_sunday,month,year,error_abs,error
10168,2025-10-29 19:00:00+00:00,4264.0,AV_Champs_Elysees,998.0,19.26056,Pré-saturé,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,0.5,-0.965926,0.974928,-0.866025,2025-10-29,0,10,2025,162.181702,-162.181702
10169,2025-10-29 20:00:00+00:00,4264.0,AV_Champs_Elysees,1041.0,20.32667,Pré-saturé,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,0.5,-0.866025,0.974928,-0.866025,2025-10-29,0,10,2025,242.786072,-242.786072
10170,2025-10-29 21:00:00+00:00,4264.0,AV_Champs_Elysees,658.0,11.17000,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,0.5,-0.707107,0.974928,-0.866025,2025-10-29,0,10,2025,88.624634,88.624634
10171,2025-10-29 22:00:00+00:00,4264.0,AV_Champs_Elysees,650.0,14.07389,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,0.5,-0.500000,0.974928,-0.866025,2025-10-29,0,10,2025,19.763977,19.763977
10172,2025-10-29 23:00:00+00:00,4264.0,AV_Champs_Elysees,510.0,10.46945,Fluide,2294.0,Av_Champs_Elysees-Washington,2293.0,Av_Champs_Elysees-Berri,...,0.5,-0.258819,0.974928,-0.866025,2025-10-29,0,10,2025,133.089294,133.089294


In [ ]:
df_ml["date"] = df_ml["Date et heure de comptage"].dt.date
df_ml["is_sunday"] = (df_ml["Date et heure de comptage"].dt.dayofweek == 6).astype(int)
df_ml["month"] = df_ml["Date et heure de comptage"].dt.month
df_ml["year"] = df_ml["Date et heure de comptage"].dt.year

df_ml["error_abs"] = np.abs(model.predict(df_ml[features]) - df_ml["Débit horaire"])
df_ml["error"] = model.predict(df_ml[features]) - df_ml["Débit horaire"]

group = (
    df_ml[
        [
            "is_sunday",
            "date",
            "month",
            "year",
            "error",
            "error_abs",
            "Débit horaire",
        ]
    ]
    .groupby(["date"])
    .mean()
)

fig = go.Figure()
fig.add_trace(
    go.Box(
        x=0 * group["Débit horaire"],
        y=group["Débit horaire"],
        boxpoints="outliers",
        name="Débit horaire",
        marker_color="lightblue",
    )
)
fig.add_trace(
    go.Box(
        x=1 + 0 * group["Débit horaire"],
        y=group["error"],
        boxpoints="outliers",
        name="Erreur",
        marker_color="lightcoral",
    )
)
fig.update_layout(
    title="Diagramme en moustache du Débit horaire",
    yaxis_title="Débit horaire",
)
fig.show()

In [32]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=group["error_abs"],
        name="Erreur",
    )
)

In [35]:
group[["Débit horaire", "error", "error_abs"]][group["error_abs"] > 100]

,Débit horaire,error,error_abs
date,,,
2025-08-19,627.571429,9.363489,103.456367
2025-08-27,808.125000,-121.756409,143.362947
2025-08-29,852.909091,-104.820386,132.299112
2025-08-30,877.916667,-138.669075,167.326413
2025-08-31,829.416667,-111.118923,124.517249
2025-09-01,770.375000,-42.824524,112.994146
2025-09-03,879.500000,-70.059968,105.816148
2025-09-21,292.434783,345.291087,345.291087
2025-09-28,558.565217,74.723999,128.726823
